# Decision-Calibrated PDE World Models
## From predictive uncertainty to control-effective dynamics ambiguity sets

This notebook is the reproducible entry point for the open-source project. It keeps three claims separate:

1. **Prediction:** perturbation disagreement plus conformal calibration gives simultaneous function-valued coverage.
2. **Decision:** support functions convert the same ambiguity set into constraint tightening or robust value penalties.
3. **Control:** closed-loop performance must be measured independently; good coverage alone does not imply good control.

Every result is exported as an individual figure rather than a composite panel.

## 0. Colab runtime and project upload

Choose a GPU runtime, then run this cell. Upload the release ZIP produced with the repository. No private token is required.

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile, torch

uploaded = files.upload()
archive_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive_name) as bundle:
    bundle.extractall('/content')
PROJECT = Path('/content/decision-calibrated-pde-control')
assert PROJECT.exists(), f'Missing release folder at {PROJECT}'
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!pip -q install -e '.[dev,ns2d]'
!python -m compileall -q src tests
print('Environment ready.')

## 1. Official two-dimensional Navier–Stokes data

The official NeuralOperator loader downloads Zenodo record `12825163`. The 128×128 archive is about 1.5 GB and contains vorticity input/output tensors for Reynolds number 500. This public dataset has no action channel, so it validates two-dimensional uncertainty calibration but **does not by itself establish a closed-loop control result**.

In [ ]:
DATA_ROOT = Path('/content/ns2d')
!python scripts/download_ns2d.py --root $DATA_ROOT --resolution 128
print(sorted(path.name for path in DATA_ROOT.glob('*.pt')))

## 2. Two-FNO uncertainty and simultaneous conformal coverage

A base FNO is trained on clean labels and a second FNO on a fixed Gaussian-perturbed copy of the same labels. Their smoothed disagreement is only an error-localization scale. A max-type conformal score supplies finite-sample simultaneous coverage over the full 128×128 output field.

In [ ]:
!dcurc-ns2d \
  --data-root $DATA_ROOT \
  --output-dir experiments/ns2d \
  --n-train 800 --n-audit 200 --n-test 300 \
  --epochs 20 --batch-size 8 --modes 16 --hidden-channels 32 \
  --label-noise 0.05 --smoothing-window 15 --no-download

The NS2D experiment writes eleven separate figures: input, target, prediction, absolute error, uncertainty width, coverage mask, training curve, score ECDF, reliability, coverage–width trade-off, and error–scale association.

## 3. Controlled Burgers world model and robust control

The controlled one-dimensional benchmark supplies the action-conditioned dynamics needed for MPC. It evaluates viscosity, boundary, actuator-gain and combined shifts. FNO is the initial world model, not the contribution.

In [ ]:
!dcurc-experiment --model fno --uncertainty perturbation --quick --control-cases 8 --control-horizon 15 --device auto --output-dir results/colab_fno_seed27 --seed 27
!dcurc-value-gap --output-root experiments/reward_value_gap --checkpoint results/colab_fno_seed27/fno_perturbation_world_model.pt

## 4. Four value-bound comparison

The comparison uses independent calibration and test trajectories. Reporting both coverage and utilization prevents a deliberately undersized bound from appearing artificially tight. `Adjoint + curvature` is an audit-calibrated second-order proxy unless a uniform Hessian bound is separately established.

In [ ]:
!python experiments/bound_comparison/run_experiment.py \
  --checkpoint results/colab_fno_seed27/fno_perturbation_world_model.pt \
  --output-root experiments/bound_comparison \
  --calibration-cases 30 --test-cases 80 --horizon 15 --gamma 0.95

## 5. Generate the complete individual-figure suite

Each output file carries one conclusion. No multi-panel composites are produced. Figures that require unavailable experiments are listed in the manifest as pending instead of being fabricated.

In [ ]:
!python scripts/build_figure_suite.py --project-root . --output-dir figures/all_individual
from IPython.display import display
from PIL import Image
for path in sorted(Path('figures/all_individual').glob('*.png')):
    print(path.name)
    display(Image.open(path))

## 6. Package results for local download

The archive contains source CSV/JSON, individual PNG/SVG/PDF figures, checkpoints, logs and the paper. The 1.5 GB NS2D raw dataset is intentionally excluded.

In [ ]:
import shutil
archive = shutil.make_archive('/content/dcurc_colab_results', 'zip', PROJECT, '.')
print(archive)
files.download(archive)